# 🌸 Iris Flower Classification — World-Class ML Pipeline
### Complete Step-by-Step Guide: Data Cleaning → EDA → Model Building → Evaluation
---
> **Dataset:** 150 samples | 3 classes (Setosa, Versicolor, Virginica) | 4 features
> 
> **Goal:** Build the most accurate multi-class classifier with full validation.

## 📦 Step 1: Install & Import All Libraries

In [ ]:
# ── Install any missing packages (Colab-safe) ──────────────────────────────
!pip install -q scikit-learn pandas numpy matplotlib seaborn plotly xgboost lightgbm

# ── Core Libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Scikit-learn: Preprocessing ─────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)

# ── Scikit-learn: Models ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier,
                               VotingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ── Scikit-learn: Metrics ─────────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# ── Style Setup ───────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})
PALETTE = ['#4C72B0', '#DD8452', '#55A868']  # Blue, Orange, Green

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load Dataset

In [ ]:
# ── Option A: Upload your IRIS.csv (Colab file upload) ─────────────────────
from google.colab import files
uploaded = files.upload()   # ← click 'Choose File' → select IRIS.csv

df = pd.read_csv('IRIS.csv')

# ── Option B: Load directly from sklearn (if no CSV) ──────────────────────
# from sklearn.datasets import load_iris
# iris = load_iris(as_frame=True)
# df = iris.frame
# df.columns = ['sepal_length','sepal_width','petal_length','petal_width','species']

print(f'Dataset Shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
display(df.head(10))

---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('=' * 60)
print('  DATASET OVERVIEW')
print('=' * 60)

print(f'\n📐 Shape         : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'🏷️  Feature Cols   : {list(df.select_dtypes(include=np.number).columns)}')
print(f'🎯 Target Column  : species')
print(f'\n📊 Class Distribution:')
print(df['species'].value_counts().to_string())

print(f'\n📈 Statistical Summary:')
display(df.describe().round(3))

# ── Data Types ─────────────────────────────────────────────────────────────
print(f'\n🗂️  Data Types:')
print(df.dtypes)

In [ ]:
# ── Visualization 1: Class Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['species'].value_counts()
axes[0].bar(counts.index, counts.values, color=PALETTE, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=PALETTE,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 11})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# ── Visualization 2: Feature Distributions (Box + Violin) ──────────────────
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i, feat in enumerate(features):
    sns.boxplot(data=df, x='species', y=feat, palette=PALETTE, ax=axes[0, i])
    axes[0, i].set_title(f'{feat} (Boxplot)', fontsize=11, fontweight='bold')
    axes[0, i].set_xlabel('')

    sns.violinplot(data=df, x='species', y=feat, palette=PALETTE, ax=axes[1, i], inner='quartile')
    axes[1, i].set_title(f'{feat} (Violin)', fontsize=11, fontweight='bold')
    axes[1, i].set_xlabel('')

plt.suptitle('Feature Distributions by Species', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Visualization 3: Pairplot ───────────────────────────────────────────────
g = sns.pairplot(df, hue='species', palette=PALETTE, diag_kind='kde',
                 plot_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot of All Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

# ── Visualization 4: Correlation Heatmap ───────────────────────────────────
plt.figure(figsize=(8, 6))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 13, 'weight': 'bold'})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🧹 Step 4: Data Cleaning & Preprocessing

In [ ]:
print('🔎 CLEANING PIPELINE')
print('=' * 50)

# ── 4.1 Duplicate Check ────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'\n[1] Duplicate rows  : {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f'    → Dropped. New shape: {df.shape}')
else:
    print('    → None found ✅')

# ── 4.2 Missing Values ─────────────────────────────────────────────────────
missing = df.isnull().sum()
print(f'\n[2] Missing values  :\n{missing.to_string()}')
if missing.sum() > 0:
    # Fill numeric columns with median (robust to outliers)
    for col in df.select_dtypes(include=np.number).columns:
        df[col].fillna(df[col].median(), inplace=True)
    print('    → Filled with column medians ✅')
else:
    print('    → None found ✅')

# ── 4.3 Outlier Detection (IQR method) ────────────────────────────────────
print(f'\n[3] Outlier Check (IQR method):')
for col in features:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f'    {col:15s}: {outliers} outlier(s)  [range: {lower:.2f} – {upper:.2f}]')

print('    → Keeping outliers (they are valid biological measurements) ✅')

# ── 4.4 Standardize Species Labels ─────────────────────────────────────────
df['species'] = df['species'].str.strip().str.lower().str.replace('iris-', '', regex=False)
print(f'\n[4] Cleaned species labels: {df["species"].unique()}')

# ── 4.5 Encode Target ──────────────────────────────────────────────────────
le = LabelEncoder()
df['species_enc'] = le.fit_transform(df['species'])
class_names = le.classes_
print(f'\n[5] Label Encoding: {dict(zip(class_names, le.transform(class_names)))}')

print(f'\n✅ Cleaned Dataset Shape: {df.shape}')
display(df.head())

---
## ✂️ Step 5: Feature Engineering & Train-Test Split

In [ ]:
# ── Features & Target ─────────────────────────────────────────────────────
X = df[features].values
y = df['species_enc'].values

# ── Stratified Train / Test Split (80/20) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f'Train size : {X_train.shape[0]} samples  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test  size : {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Class balance (train): {np.bincount(y_train)}')
print(f'Class balance (test) : {np.bincount(y_test)}')

# ── Feature Scaling (StandardScaler fitted on train only) ─────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit+transform on train
X_test_sc  = scaler.transform(X_test)        # transform only on test

print(f'\n✅ Scaling done. Feature means (train): {X_train_sc.mean(axis=0).round(4)}')
print(f'   Feature stds  (train): {X_train_sc.std(axis=0).round(4)}')

---
## 🤖 Step 6: Train & Compare Multiple Models

In [ ]:
# ── Define all candidate models ────────────────────────────────────────────
models = {
    'Logistic Regression'    : LogisticRegression(max_iter=1000, random_state=42),
    'KNN'                    : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'          : DecisionTreeClassifier(random_state=42),
    'Random Forest'          : RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees'            : ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'      : GradientBoostingClassifier(random_state=42),
    'AdaBoost'               : AdaBoostClassifier(random_state=42),
    'XGBoost'                : XGBClassifier(eval_metric='mlogloss', random_state=42, verbosity=0),
    'LightGBM'               : LGBMClassifier(random_state=42, verbose=-1),
    'SVM (RBF)'              : SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes'            : GaussianNB(),
    'LDA'                    : LinearDiscriminantAnalysis(),
}

# ── Cross-validation setup (5-fold Stratified) ────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
print(f'{"Model":<25}  {"CV Mean":>9}  {"CV Std":>8}  {"Test Acc":>9}')
print('─' * 60)

for name, model in models.items():
    # 5-fold CV on scaled training data
    cv_scores = cross_val_score(model, X_train_sc, y_train,
                                cv=skf, scoring='accuracy', n_jobs=-1)
    # Fit on full train, predict on test
    model.fit(X_train_sc, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test_sc))

    results[name] = {
        'cv_mean' : cv_scores.mean(),
        'cv_std'  : cv_scores.std(),
        'test_acc': test_acc,
        'model'   : model
    }
    print(f'{name:<25}  {cv_scores.mean():.5f}    {cv_scores.std():.5f}    {test_acc:.5f}')

print('─' * 60)

# ── Best model by test accuracy ────────────────────────────────────────────
best_name = max(results, key=lambda k: results[k]['test_acc'])
best_model = results[best_name]['model']
print(f'\n🏆 Best Model: {best_name}  |  Test Accuracy: {results[best_name]["test_acc"]:.5f}')

In [ ]:
# ── Bar chart: CV mean accuracy across all models ─────────────────────────
res_df = pd.DataFrame(results).T[['cv_mean', 'cv_std', 'test_acc']]
res_df = res_df.sort_values('test_acc', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#2ecc71' if n == best_name else '#3498db' for n in res_df.index]
bars = ax.barh(res_df.index, res_df['test_acc'], xerr=res_df['cv_std'],
               color=colors, edgecolor='black', linewidth=0.7, capsize=4)
for bar, val in zip(bars, res_df['test_acc']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy', fontsize=13)
ax.set_title('Model Comparison — Test Accuracy', fontsize=15, fontweight='bold')
ax.set_xlim(0.8, 1.05)
green_patch = mpatches.Patch(color='#2ecc71', label='Best Model')
ax.legend(handles=[green_patch], loc='lower right')
plt.tight_layout()
plt.show()

---
## ⚙️ Step 7: Hyperparameter Tuning (Best Model)

In [ ]:
# ── GridSearchCV on Random Forest (tunable to your best model) ────────────
param_grid = {
    'n_estimators'     : [50, 100, 200],
    'max_depth'        : [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2],
    'max_features'     : ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=skf, scoring='accuracy',
                           n_jobs=-1, verbose=1)
grid_search.fit(X_train_sc, y_train)

tuned_model = grid_search.best_estimator_
tuned_acc   = accuracy_score(y_test, tuned_model.predict(X_test_sc))

print(f'\n🔧 Best Params : {grid_search.best_params_}')
print(f'📊 CV Best Score : {grid_search.best_score_:.5f}')
print(f'🎯 Test Accuracy : {tuned_acc:.5f}')

---
## 📊 Step 8: Final Model Evaluation

In [ ]:
# Use tuned_model as final model
final_model = tuned_model
y_pred      = final_model.predict(X_test_sc)
y_prob      = final_model.predict_proba(X_test_sc)

# ── Classification Report ──────────────────────────────────────────────────
print('📋 CLASSIFICATION REPORT')
print('─' * 50)
print(classification_report(y_test, y_pred, target_names=class_names))

# ── Confusion Matrix ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ── Normalised Confusion Matrix ───────────────────────────────────────────
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2   = ConfusionMatrixDisplay(confusion_matrix=cm_norm.round(2), display_labels=class_names)
disp2.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Normalised Confusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# ── Multi-class ROC AUC ────────────────────────────────────────────────────
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
roc_auc    = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro')
print(f'\n🎯 Final Test Accuracy : {accuracy_score(y_test, y_pred):.5f}')
print(f'📐 Macro ROC-AUC       : {roc_auc:.5f}')

In [ ]:
# ── ROC Curves (One-vs-Rest) ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc_i   = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=PALETTE[i],
            label=f'{cls} (AUC = {roc_auc_i:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlim([-0.02, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — One vs Rest', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

---
## 🌲 Step 9: Feature Importance & PCA Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Feature Importance (Random Forest) ────────────────────────────────────
importances = final_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

axes[0].bar(range(len(features)),
            importances[sorted_idx],
            color=['#e74c3c','#3498db','#2ecc71','#f39c12'],
            edgecolor='black', linewidth=0.8)
axes[0].set_xticks(range(len(features)))
axes[0].set_xticklabels([features[i] for i in sorted_idx], rotation=20)
axes[0].set_title('Feature Importance', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Importance Score')
for i, v in enumerate(importances[sorted_idx]):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

# ── PCA 2D Visualization ──────────────────────────────────────────────────
pca     = PCA(n_components=2, random_state=42)
X_pca   = pca.fit_transform(scaler.transform(X))
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y,
                          cmap='Set1', edgecolors='k', linewidth=0.4, alpha=0.8, s=60)
handles, _ = scatter.legend_elements()
axes[1].legend(handles, class_names, title='Species', fontsize=10)
axes[1].set_title(f'PCA 2D  (Explained Var: {pca.explained_variance_ratio_.sum()*100:.1f}%)',
                   fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.tight_layout()
plt.show()

---
## 🔮 Step 10: Predict on New Data

In [ ]:
# ── Provide new flower measurements [sepal_length, sepal_width, petal_length, petal_width]
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],   # Expected: setosa
    [6.0, 2.9, 4.5, 1.5],   # Expected: versicolor
    [6.7, 3.0, 5.2, 2.3],   # Expected: virginica
])

new_scaled   = scaler.transform(new_flowers)
predictions  = final_model.predict(new_scaled)
probabilities = final_model.predict_proba(new_scaled)

print('🌸 NEW FLOWER PREDICTIONS')
print('─' * 65)
print(f'{"#":<4} {"Measurements":<32} {"Predicted":<15} {"Confidence"}')
print('─' * 65)

for i, (row, pred, prob) in enumerate(zip(new_flowers, predictions, probabilities)):
    species  = class_names[pred]
    conf     = prob.max() * 100
    meas_str = str(row.tolist())
    print(f'{i+1:<4} {meas_str:<32} {species:<15} {conf:.2f}%')

print('─' * 65)

---
## 💾 Step 11: Save the Model

In [ ]:
import joblib

# ── Save model & scaler ────────────────────────────────────────────────────
joblib.dump(final_model, 'iris_best_model.pkl')
joblib.dump(scaler,      'iris_scaler.pkl')
joblib.dump(le,          'iris_label_encoder.pkl')

print('✅ Model saved: iris_best_model.pkl')
print('✅ Scaler saved: iris_scaler.pkl')
print('✅ LabelEncoder saved: iris_label_encoder.pkl')

# ── Download from Colab ─────────────────────────────────────────────────────
from google.colab import files
files.download('iris_best_model.pkl')
files.download('iris_scaler.pkl')

---
## 📋 Step 12: Final Summary

In [ ]:
print('=' * 55)
print('        🌸  IRIS CLASSIFICATION — FINAL SUMMARY  🌸')
print('=' * 55)
print(f'  Dataset            : 150 samples, 4 features, 3 classes')
print(f'  Duplicates removed : {dupes}')
print(f'  Missing values     : 0')
print(f'  Train/Test Split   : 80% / 20% (stratified)')
print(f'  Scaling            : StandardScaler')
print(f'  Models compared    : {len(models)}')
print(f'  Best Base Model    : {best_name}  ({results[best_name]["test_acc"]:.5f})')
print(f'  After Tuning       : {tuned_acc:.5f}')
print(f'  Macro ROC-AUC      : {roc_auc:.5f}')
print('=' * 55)